# Randomisation Inference Simulation

Compares two inferential approaches for each assignment design:

| Approach | SE source | Valid for BWD? |
|---|---|---|
| **Neyman** | Sample variances (ignores design) | Conservative — overcoverage, lower power |
| **Randomisation Inference** | Re-run BWD B times | Exact — exploits known assignment mechanism |

Metrics measured per design per iteration:
- `NeymanCoverage` / `RI_Coverage` — does the 95% CI contain the true ATE?
- `NeymanRejects` / `RI_Rejects` — do we reject H0: tau=0 when a real effect exists?
- `ATEError` — bias of the point estimate
- `CovariateMSE` — covariate imbalance between groups

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import evaluator as evl
from alias import bal, dgp, est
from plan import Plan

In [ ]:
# ── Simulation parameters ─────────────────────────────────────────────────────
B = 500  # RI re-runs per experiment
NUM_ITERS = 100  # repetitions per (design, sample_size, dgp) cell
# increase to 500 for final thesis results

sample_sizes = [100, 250, 500, 1000]

In [ ]:
# ── Build the plan ────────────────────────────────────────────────────────────
# The RI evaluators are registered once with balancer_class=bal.BWD.
# They re-run BWD internally regardless of which design ran the real experiment,
# so we can compare RI against Neyman across all designs on equal footing.

bwd_kwargs = {"phi": 1.0, "delta": 0.05}

plan = Plan()

# Designs
plan.add_design("Simple", bal.Simple, est.DifferenceInMeans, {})
plan.add_design("BWD(1)", bal.BWD, est.DifferenceInMeans, {"phi": 1.0})
plan.add_design("BWD(0.5)", bal.BWD, est.DifferenceInMeans, {"phi": 0.5})

# Standard metrics
plan.add_evaluator("ATEError", evl.ATEError)
plan.add_evaluator("CovariateMSE", evl.CovariateMSE)
plan.add_evaluator("CISize", evl.CISize)

# Neyman inference (baseline)
plan.add_evaluator("NeymanCoverage", evl.ATECovers)  # fixed bug
plan.add_evaluator("NeymanRejects", evl.NeymanRejects)

# Randomisation inference
ri_kwargs = {"balancer_class": bal.BWD, "balancer_kwargs": bwd_kwargs, "B": B}
plan.add_evaluator("RI_PValue", evl.RandomisationInferencePValue, ri_kwargs)
plan.add_evaluator("RI_Rejects", evl.RandomisationInferenceRejects, ri_kwargs)
plan.add_evaluator("RI_Coverage", evl.RandomisationInferenceCoverage, ri_kwargs)

In [ ]:
# ── Run the simulation ────────────────────────────────────────────────────────
# Start with LinearDGP only. Extend to other DGPs once results look sensible.

os.makedirs("../results/inference", exist_ok=True)

dfs = []
dgp_factories = [dgp.LinearFactory]

for sample_size in sample_sizes:
    print(f"\nSample size: {sample_size}", flush=True)
    for factory_class in dgp_factories:
        factory = factory_class(N=sample_size)
        dgp_name = type(factory.create_dgp()).__name__
        print(f"  DGP: {dgp_name}", flush=True)
        for it in tqdm(range(NUM_ITERS)):
            result = plan.execute(factory, seed=it * 1001)
            result["iteration"] = it
            result["sample_size"] = sample_size
            result["dgp"] = dgp_name
            dfs.append(result)

results = pd.concat(dfs, ignore_index=True)
results.to_csv("../results/inference/linear_results.csv.gz", index=False)
print("\nDone. Results saved to results/inference/linear_results.csv.gz")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary = (
    results.groupby(["dgp", "sample_size", "design", "metric"])["value"]
    .agg(mean="mean", se=lambda x: x.std() / np.sqrt(len(x)))
    .reset_index()
)

key_metrics = [
    "NeymanCoverage",
    "RI_Coverage",
    "NeymanRejects",
    "RI_Rejects",
    "CISize",
    "CovariateMSE",
]
(
    summary[summary["metric"].isin(key_metrics)]
    .pivot_table(index=["sample_size", "design"], columns="metric", values="mean")
    .round(3)
)

In [ ]:
# ── Coverage and power plots ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

designs = results["design"].unique()
colors = plt.cm.tab10.colors
design_color = dict(zip(designs, colors, strict=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("LinearDGP — Neyman vs Randomisation Inference", fontsize=13)

for ax, (neyman_m, ri_m, title, target) in zip(
    axes,
    [
        ("NeymanCoverage", "RI_Coverage", "Coverage (target = 0.95)", 0.95),
        ("NeymanRejects", "RI_Rejects", "Power (true ATE = 1)", None),
    ],
    strict=False,
):
    for design in designs:
        color = design_color[design]
        for metric, ls, label_suffix in [
            (neyman_m, "-", "(Neyman)"),
            (ri_m, "--", "(RI)"),
        ]:
            sub = summary[(summary["metric"] == metric) & (summary["design"] == design)]
            ax.plot(
                sub["sample_size"],
                sub["mean"],
                color=color,
                linestyle=ls,
                marker="o",
                label=f"{design} {label_suffix}",
            )
    if target:
        ax.axhline(target, color="black", linestyle=":", linewidth=1, label="target")
    ax.set_title(title)
    ax.set_xlabel("Sample size")
    ax.set_xscale("log")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(
    "../results/figures/inference_coverage_power.png", dpi=150, bbox_inches="tight"
)
plt.show()